# 01 — Data Ingestion

## Purpose

This notebook **reproduces and tests the current production data-ingestion logic** before we upgrade it.

Current production flow:

```text
MongoDB Collection
      ↓
export_collection_as_dataframe()
      ↓
Drop MongoDB `_id`
Replace `"na"` → NaN
      ↓
Feature Store CSV
      ↓
train_test_split(test_size=0.2, random_state=42)
      ↓
train.csv + test.csv
      ↓
DataIngestionArtifact
```

> **Important:** This notebook is intentionally a laboratory mirror of the current pipeline. Do not move experimental changes into `src/network_security/components/data_ingestion.py` until they are validated here.


## 1. Current implementation inspected

The current `DataIngestion` component:

- Reads database and collection names from `DataIngestionConfig`.
- Connects using `MONGO_DB_URL`.
- Loads the entire MongoDB collection into memory using `list(collection.find())`.
- Drops MongoDB's automatically generated `_id` column.
- Replaces the literal string `"na"` with `np.nan`.
- Writes the complete dataset to the feature store.
- Splits data with:
  - `test_size = 0.2`
  - `random_state = 42`
  - **no stratification currently enabled**
- Writes `train.csv` and `test.csv`.
- Returns only the two file paths through `DataIngestionArtifact`.

This notebook first verifies that behavior using the project's local raw CSV as a reproducible fallback when MongoDB is unavailable.


In [1]:
from pathlib import Path
import sys
import os
import json
from dataclasses import dataclass
from datetime import datetime

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)


## 2. Locate the project root

In [2]:
def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()

    for candidate in [start, *start.parents]:
        if (candidate / "src").exists() and (candidate / "data").exists():
            return candidate

    raise FileNotFoundError(
        "Project root not found. Open this notebook from inside the project repository."
    )

PROJECT_ROOT = find_project_root()
PROJECT_ROOT


WindowsPath('e:/Projects/Network security log triage agent')

In [3]:
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
print(f"Python executable: {sys.executable}")


Project root: e:\Projects\Network security log triage agent
Python executable: e:\Projects\Network security log triage agent\.venv\Scripts\python.exe


## 3. Inspect the current ingestion configuration

In [ ]:
# Mirror of the current constants/configuration.
TARGET_COLUMN = "Result"

PIPELINE_NAME = "NetworkSecurity"
ARTIFACT_DIR = "Artifacts"
FILE_NAME = "phisingData.csv"

TRAIN_FILE_NAME = "train.csv"
TEST_FILE_NAME = "test.csv"

DATA_INGESTION_COLLECTION_NAME = "NetworkData"
DATA_INGESTION_DATABASE_NAME = "Jahid'sAI"
DATA_INGESTION_DIR_NAME = "data_ingestion"
DATA_INGESTION_FEATURE_STORE_DIR = "feature_store"
DATA_INGESTION_INGESTED_DIR = "ingested"
DATA_INGESTION_TRAIN_TEST_SPLIT_RATIO = 0.2

RANDOM_STATE = 42


In [ ]:
class TrainingPipelineConfig:
    def __init__(self, timestamp=datetime.now()):
        timestamp = timestamp.strftime("%m_%d_%Y_%H_%M_%S")
        self.pipeline_name = PIPELINE_NAME
        self.artifact_name = ARTIFACT_DIR
        self.artifact_dir = PROJECT_ROOT / self.artifact_name / timestamp
        self.model_dir = PROJECT_ROOT / "final_model"
        self.timestamp = timestamp


class DataIngestionConfig:
    def __init__(self, training_pipeline_config):
        self.data_ingestion_dir = (
            training_pipeline_config.artifact_dir / DATA_INGESTION_DIR_NAME
        )

        self.feature_store_file_path = (
            self.data_ingestion_dir
            / DATA_INGESTION_FEATURE_STORE_DIR
            / FILE_NAME
        )

        self.training_file_path = (
            self.data_ingestion_dir
            / DATA_INGESTION_INGESTED_DIR
            / TRAIN_FILE_NAME
        )

        self.testing_file_path = (
            self.data_ingestion_dir
            / DATA_INGESTION_INGESTED_DIR
            / TEST_FILE_NAME
        )

        self.train_test_split_ratio = DATA_INGESTION_TRAIN_TEST_SPLIT_RATIO
        self.collection_name = DATA_INGESTION_COLLECTION_NAME
        self.database_name = DATA_INGESTION_DATABASE_NAME


training_pipeline_config = TrainingPipelineConfig()
data_ingestion_config = DataIngestionConfig(training_pipeline_config)

print("Timestamp:", training_pipeline_config.timestamp)
print("Feature store:", data_ingestion_config.feature_store_file_path)
print("Train file:", data_ingestion_config.training_file_path)
print("Test file:", data_ingestion_config.testing_file_path)
print("Database:", data_ingestion_config.database_name)
print("Collection:", data_ingestion_config.collection_name)


## 4. Data source

The production pipeline currently uses MongoDB.

For notebook reproducibility, this section supports two modes:

1. **MongoDB** — reproduces production ingestion.
2. **Local CSV fallback** — allows the notebook to run without exposing database credentials.

The project's current raw dataset is:

```text
data/raw/phisingData.csv
```


In [ ]:
DATA_SOURCE = "local_csv"  # Change to "mongodb" when MongoDB access is configured.

RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "phisingData.csv"
MONGO_DB_URL = os.getenv("MONGO_DB_URL")

print("Data source:", DATA_SOURCE)
print("Raw CSV exists:", RAW_DATA_PATH.exists())
print("MongoDB URL configured:", bool(MONGO_DB_URL))


In [ ]:
def export_collection_as_dataframe_mongodb(config):
    """Mirror the current production MongoDB ingestion logic."""
    try:
        import pymongo
    except ImportError as e:
        raise ImportError(
            "pymongo is required for MongoDB ingestion. Install project dependencies first."
        ) from e

    if not MONGO_DB_URL:
        raise ValueError("MONGO_DB_URL is not configured.")

    mongo_client = pymongo.MongoClient(MONGO_DB_URL)

    collection = mongo_client[
        config.database_name
    ][
        config.collection_name
    ]

    dataframe = pd.DataFrame(list(collection.find()))

    if "_id" in dataframe.columns:
        dataframe = dataframe.drop(columns=["_id"], axis=1)

    dataframe.replace({"na": np.nan}, inplace=True)

    return dataframe


def export_collection_as_dataframe_local(file_path):
    """Reproducible notebook fallback using the project's current raw CSV."""
    dataframe = pd.read_csv(file_path)

    if "_id" in dataframe.columns:
        dataframe = dataframe.drop(columns=["_id"], axis=1)

    dataframe.replace({"na": np.nan}, inplace=True)

    return dataframe


In [ ]:
if DATA_SOURCE == "mongodb":
    dataframe = export_collection_as_dataframe_mongodb(data_ingestion_config)
elif DATA_SOURCE == "local_csv":
    dataframe = export_collection_as_dataframe_local(RAW_DATA_PATH)
else:
    raise ValueError("DATA_SOURCE must be either 'mongodb' or 'local_csv'.")

print("Data shape:", dataframe.shape)
display(dataframe.head())


## 5. Initial ingestion inspection

In [ ]:
summary = pd.DataFrame({
    "dtype": dataframe.dtypes.astype(str),
    "missing_values": dataframe.isna().sum(),
    "unique_values": dataframe.nunique(dropna=False),
})

display(summary)


In [ ]:
print("Duplicate rows:", dataframe.duplicated().sum())
print("Columns:", dataframe.shape[1])
print("Rows:", dataframe.shape[0])

if TARGET_COLUMN in dataframe.columns:
    print("\nTarget distribution:")
    display(
        dataframe[TARGET_COLUMN]
        .value_counts(dropna=False)
        .rename_axis(TARGET_COLUMN)
        .to_frame("count")
    )
else:
    print(f"WARNING: Target column '{TARGET_COLUMN}' was not found.")


## 6. Reproduce `export_data_into_feature_store`

The current implementation creates the parent directory and writes the complete dataframe to CSV without the index.


In [ ]:
def export_data_into_feature_store(dataframe, config):
    feature_store_file_path = Path(config.feature_store_file_path)

    feature_store_file_path.parent.mkdir(parents=True, exist_ok=True)

    dataframe.to_csv(
        feature_store_file_path,
        index=False,
        header=True,
    )

    return dataframe


feature_store_dataframe = export_data_into_feature_store(
    dataframe=dataframe,
    config=data_ingestion_config,
)

print("Feature store written to:")
print(data_ingestion_config.feature_store_file_path)
print("Exists:", Path(data_ingestion_config.feature_store_file_path).exists())


In [ ]:
feature_store_check = pd.read_csv(data_ingestion_config.feature_store_file_path)

assert feature_store_check.shape == dataframe.shape, (
    f"Shape mismatch: expected {dataframe.shape}, got {feature_store_check.shape}"
)

assert feature_store_check.columns.tolist() == dataframe.columns.tolist(), (
    "Column mismatch between source dataframe and feature store."
)

print("Feature store verification: PASSED")
print("Shape:", feature_store_check.shape)


## 7. Reproduce the current train/test split

The production code currently uses:

```python
train_test_split(
    dataframe,
    test_size=0.2,
    random_state=42,
    # stratify=...  # currently disabled
)
```

We deliberately keep the current behavior unchanged in this baseline notebook.


In [ ]:
def split_data_as_train_test(dataframe, config):
    train_set, test_set = train_test_split(
        dataframe,
        test_size=config.train_test_split_ratio,
        random_state=RANDOM_STATE,
        # stratify=dataframe[TARGET_COLUMN]  # intentionally disabled to mirror current pipeline
    )

    training_file_path = Path(config.training_file_path)
    testing_file_path = Path(config.testing_file_path)

    training_file_path.parent.mkdir(parents=True, exist_ok=True)

    train_set.to_csv(
        training_file_path,
        index=False,
        header=True,
    )

    test_set.to_csv(
        testing_file_path,
        index=False,
        header=True,
    )

    return train_set, test_set


train_set, test_set = split_data_as_train_test(
    dataframe=feature_store_dataframe,
    config=data_ingestion_config,
)

print("Train shape:", train_set.shape)
print("Test shape:", test_set.shape)
print("Train path:", data_ingestion_config.training_file_path)
print("Test path:", data_ingestion_config.testing_file_path)


## 8. Verify split reproducibility

In [ ]:
train_set_2, test_set_2 = train_test_split(
    dataframe,
    test_size=data_ingestion_config.train_test_split_ratio,
    random_state=RANDOM_STATE,
)

assert train_set.index.equals(train_set_2.index)
assert test_set.index.equals(test_set_2.index)

print("Reproducibility check: PASSED")


## 9. Inspect class distribution before and after splitting

In [ ]:
if TARGET_COLUMN in dataframe.columns:
    def distribution_frame(series):
        counts = series.value_counts(dropna=False)
        return pd.DataFrame({
            "count": counts,
            "ratio": (counts / counts.sum()).round(4),
        })

    print("Full dataset")
    display(distribution_frame(dataframe[TARGET_COLUMN]))

    print("Train set")
    display(distribution_frame(train_set[TARGET_COLUMN]))

    print("Test set")
    display(distribution_frame(test_set[TARGET_COLUMN]))


## 10. Validate generated artifacts

The current `DataIngestionArtifact` only returns the generated train and test file paths.


In [ ]:
@dataclass
class DataIngestionArtifact:
    trained_file_path: str
    test_file_path: str


data_ingestion_artifact = DataIngestionArtifact(
    trained_file_path=str(data_ingestion_config.training_file_path),
    test_file_path=str(data_ingestion_config.testing_file_path),
)

data_ingestion_artifact


In [ ]:
assert Path(data_ingestion_artifact.trained_file_path).exists()
assert Path(data_ingestion_artifact.test_file_path).exists()

train_check = pd.read_csv(data_ingestion_artifact.trained_file_path)
test_check = pd.read_csv(data_ingestion_artifact.test_file_path)

assert len(train_check) + len(test_check) == len(dataframe)
assert set(train_check.columns) == set(dataframe.columns)
assert set(test_check.columns) == set(dataframe.columns)

print("Artifact validation: PASSED")
print(f"Total rows: {len(dataframe)}")
print(f"Train rows: {len(train_check)}")
print(f"Test rows: {len(test_check)}")


# 11. Current pipeline behavior — findings

This notebook should initially remain a **baseline reproduction**. Based on the inspected implementation, the main limitations to investigate later are:

### A. Entire MongoDB collection is loaded into memory

```python
pd.DataFrame(list(collection.find()))
```

This is simple but may become unsuitable for large security-log collections.

### B. No explicit connection verification or timeout configuration

The current client uses:

```python
pymongo.MongoClient(MONGO_DB_URL)
```

without explicit connection checks, timeouts, or controlled cleanup.

### C. `"na"` replacement is limited

Only the exact string `"na"` is converted to `NaN`.

### D. No schema checks during ingestion

Column count, expected columns, target existence, and data types are not verified here.

### E. No duplicate checks

Duplicate records are passed directly into the feature store.

### F. No stratified split

For classification data, the current target distribution may shift between train and test sets because stratification is disabled.

### G. Hard-coded random state

`42` is embedded in the component instead of being centrally configurable.

### H. No ingestion metadata

The artifact contains only file paths. Useful future metadata could include:

- source type
- row count
- column count
- dataset fingerprint
- ingestion timestamp
- train/test sizes
- class distribution
- dataset version


## 12. Baseline summary

At this point we have reproduced the current ingestion pipeline:

```text
Source
  ↓
DataFrame
  ↓
Drop `_id`
  ↓
Replace `"na"` → NaN
  ↓
Feature Store
  ↓
80/20 Random Split
  ↓
train.csv + test.csv
  ↓
DataIngestionArtifact
```

**Do not upgrade the production component yet.**

The next experiments should be added below as separate sections so we can compare each proposed improvement against this baseline.


# 13. Experiment area — keep future changes below

Suggested experiment order:

1. Compare **random split vs stratified split**.
2. Add ingestion-time dataset profiling.
3. Add schema and target checks without duplicating later validation responsibilities.
4. Test dataset fingerprinting.
5. Test duplicate detection and handling strategy.
6. Test MongoDB connection timeouts and explicit connectivity checks.
7. Test chunked/streaming ingestion for larger log datasets.
8. Design a richer `DataIngestionArtifact` with metadata.

Each experiment should answer:

```text
Hypothesis
    ↓
Implementation
    ↓
Benchmark / Validation
    ↓
Result
    ↓
Keep or Reject
```


In [ ]:
# ==============================
# FUTURE EXPERIMENTS START HERE
# ==============================

# Add experimental code below.
# Keep the baseline cells above unchanged so we always have a reference implementation.
